# Data Quality EDA (Amazon Fashion)

This notebook inspects basic data availability and distributions in `data/products.parquet` without loading the entire file into memory.

It computes per-column non-null ratios via row-group iteration, and samples records to inspect examples (e.g., items with/without price).


## Notes
- The non-null summary iterates row-groups to avoid loading the full table.
- Sampling is proportional by row-group size to approximate unbiased sampling.
- Adjust `SAMPLE_TARGET` to trade off speed vs detail.
- If you want richer visuals, consider enabling `tqdm` and more plots.


## Load full parquet into memory (entire DataFrame)

Warning: this will load the entire `products.parquet` into RAM. Ensure you have sufficient memory for the file size. On Windows/Conda, 16GB+ is recommended for large datasets.


In [4]:
# Initialization: imports and paths
import os
# Avoid potential OpenMP/MKL threading hangs on Windows
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")

import json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

DATA_DIR = Path('../data')
PARQUET_PATH = DATA_DIR / 'products.parquet'
META_PATH = DATA_DIR / 'meta.json'

assert PARQUET_PATH.exists(), f"Missing file: {PARQUET_PATH}"
print('Parquet path:', PARQUET_PATH)
if META_PATH.exists():
  meta = json.loads(META_PATH.read_text())
  print('Meta rows (if any):', meta.get('row_count'))
  print('Meta columns (if any):', meta.get('columns', [])[:10])


Parquet path: ..\data\products.parquet
Meta rows (if any): 826108
Meta columns (if any): ['id', 'asin', 'parent_asin', 'main_category', 'title', 'brand', 'categories', 'features', 'description', 'description_list']


In [1]:
import pandas as pd

In [24]:
# Read full table
full_df = pd.read_parquet(PARQUET_PATH, engine='pyarrow')
print('Full shape:', full_df.shape)
mem_bytes = full_df.memory_usage(deep=True).sum()
print('Approx memory usage:', round(mem_bytes/1024/1024, 2), 'MB')
full_df.head(3)


Full shape: (826108, 23)
Approx memory usage: 1712.12 MB


,id,asin,parent_asin,main_category,title,brand,categories,features,description,description_list,...,rating_count,rating_number,image_url,product_url,images,videos,store,details,bought_together,text_for_embedding
0,B08BHN9PK5,B08BHN9PK5,B08BHN9PK5,AMAZON FASHION,YUEDGE 5 Pairs Men's Moisture Control Cushione...,YUEDGE 5 Pairs,[],[],None,[],...,NaN,NaN,https://m.media-amazon.com/images/I/41+cCfaVOF...,https://www.amazon.com/dp/B08BHN9PK5,[https://m.media-amazon.com/images/I/81XlFXImF...,[],GiveGift,"{""Package Dimensions"": ""10.31 x 8.5 x 1.73 inc...",[],YUEDGE 5 Pairs Men's Moisture Control Cushione...
1,B08R39MRDW,B08R39MRDW,B08R39MRDW,AMAZON FASHION,DouBCQ Women's Palazzo Lounge Wide Leg Casual ...,DouBCQ,[],"[Drawstring closure, Machine Wash]",Drawstring closure Machine Wash,"[Drawstring closure, Machine Wash]",...,NaN,NaN,https://m.media-amazon.com/images/I/515cR-ta1E...,https://www.amazon.com/dp/B08R39MRDW,[https://m.media-amazon.com/images/I/91Z4G2jlF...,[],DouBCQ,"{""Package Dimensions"": ""15 x 10.2 x 0.4 inches...",[],DouBCQ Women's Palazzo Lounge Wide Leg Casual ...
2,B077KJHCJ4,B077KJHCJ4,B077KJHCJ4,AMAZON FASHION,Pastel by Vivienne Honey Vanilla Girls' Trapez...,Pastel,[],"[Zipper closure, Hand Wash Only]",Zipper closure Hand Wash Only,"[Zipper closure, Hand Wash Only]",...,NaN,NaN,https://m.media-amazon.com/images/I/31GwmwNCdA...,https://www.amazon.com/dp/B077KJHCJ4,[https://m.media-amazon.com/images/I/612-CtsBA...,[],Pastel by Vivienne,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...",[],Pastel by Vivienne Honey Vanilla Girls' Trapez...


In [26]:
# Non-null summary across ALL rows for all columns
nn = full_df.notna().sum().rename('non_null').to_frame()
nn['rows'] = len(full_df)
nn['null'] = nn['rows'] - nn['non_null']
nn['non_null_%'] = (nn['non_null'] / nn['rows'] * 100).round(2)
summary_full = nn.reset_index(names='column').sort_values('non_null_%', ascending=True)
summary_full.head(20)


,column,non_null,rows,null,non_null_%
14,rating_number,0,826108,826108,0.00
13,rating_count,0,826108,826108,0.00
10,price,50249,826108,775859,6.08
8,description,469449,826108,356659,56.83
5,brand,794315,826108,31793,96.15
19,store,799267,826108,26841,96.75
4,title,826050,826108,58,99.99
0,id,826108,826108,0,100.00
20,details,826108,826108,0,100.00
18,videos,826108,826108,0,100.00


In [23]:
# Full-dataset price and rating stats
price = pd.to_numeric(full_df.get('price'), errors='coerce') if 'price' in full_df.columns else None
avg_rating = pd.to_numeric(full_df.get('average_rating'), errors='coerce') if 'average_rating' in full_df.columns else None
rating_num = pd.to_numeric(full_df.get('rating_number'), errors='coerce') if 'rating_number' in full_df.columns else None

stats_full = {}
if price is not None:
  stats_full.update({
    'price_non_null_%': float((price.notna().mean()*100).round(2)),
    'price_mean': float(price.mean()) if price.notna().any() else None,
    'price_median': float(price.median()) if price.notna().any() else None,
    'price_p95': float(price.quantile(0.95)) if price.notna().any() else None,
  })
if avg_rating is not None:
  stats_full.update({
    'avg_rating_non_null_%': float((avg_rating.notna().mean()*100).round(2)),
    'avg_rating_mean': float(avg_rating.mean()) if avg_rating.notna().any() else None,
  })
if rating_num is not None:
  stats_full.update({
    'rating_number_non_null_%': float((rating_num.notna().mean()*100).round(2)),
    'rating_number_mean': float(rating_num.mean()) if rating_num.notna().any() else None,
  })

stats_full


{'price_non_null_%': 6.08,
 'price_mean': 40.79592927222433,
 'price_median': 19.89,
 'price_p95': 127.49599999999998,
 'avg_rating_non_null_%': 100.0,
 'avg_rating_mean': 3.9106595021474204,
 'rating_number_non_null_%': 0.0,
 'rating_number_mean': None}

In [ ]:
# Rows missing price (full dataset)
if 'full_df' not in locals():
  full_df = pd.read_parquet(PARQUET_PATH, engine='pyarrow')

if 'price' not in locals():
  price = pd.to_numeric(full_df.get('price'), errors='coerce') if 'price' in full_df.columns else None

if isinstance(price, pd.Series):
  missing_price_df = full_df[price.isna()].head(100)
  cols = ['id','title','brand','price','image_url']
  if set(cols).issubset(missing_price_df.columns):
    missing_price_df[cols].head(20)
  else:
    missing_price_df.head(20)
else:
  print('No price column in dataframe.')
